[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module2_DataSimilarity/04_DimensionReduction_PCA.ipynb#copy=true)


# Dimension Reduction: Principal Component Analysis

## Learning objectives

By the end of this lesson, you should be able to:

- Explain why high-dimensional data can be difficult to visualize and model.
- Use PCA to project centered data to fewer dimensions.
- Interpret explained variance and a two-dimensional PCA visualization.
- Describe the connection between PCA, SVD, and principal-components regression.

**Student Learning Outcome (SLO 5):**
> Use PCA to reduce a data matrix and justify a component choice with explained variance.


## Why reduce dimension?

An image with $28\times28$ pixels has 784 features, and many features may be correlated. **Principal component analysis (PCA)** finds orthogonal directions of greatest variation. After centering $X$, its directions are related to the singular value decomposition $X = U\Sigma V^T$. Keeping the first few directions creates a lower-dimensional approximation.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_diabetes
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

digits = load_digits()
X_digits, y_digits = digits.data, digits.target
fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for ax, image, label in zip(axes, digits.images[:5], y_digits[:5]):
    ax.imshow(image, cmap="gray_r"); ax.set_title(label); ax.axis("off")


## Part 1 — PCA visualization

We scale pixels and project each image to two components. A point’s location summarizes the image; it is not a complete reconstruction.


In [ ]:
pca_2d = PCA(n_components=2, random_state=232)
coordinates = pca_2d.fit_transform(StandardScaler().fit_transform(X_digits))

plt.figure(figsize=(8, 6))
scatter = plt.scatter(coordinates[:, 0], coordinates[:, 1], c=y_digits, s=10, cmap="tab10")
plt.colorbar(scatter, label="digit label")
plt.xlabel("principal component 1"); plt.ylabel("principal component 2"); plt.show()
print("Explained variance by two components:", pca_2d.explained_variance_ratio_.sum().round(3))


### ✏️ Written response 1

Which digit labels appear well-separated in the plot, and which overlap? Why would overlap matter if we later used the two coordinates for classification?

> **YOUR ANSWER:**


## Part 2 — Choosing components and principal-components regression

Explained variance measures how much of the feature variation the retained components preserve. In **principal-components regression (PCR)**, we first apply PCA to predictors and then fit a regression model using the retained components. The number of components should be selected using training data or cross-validation—not by looking at test results.


In [ ]:
diabetes = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.25, random_state=232
)

for n_components in [2, 5, 10]:
    pcr = make_pipeline(StandardScaler(), PCA(n_components=n_components), LinearRegression())
    pcr.fit(X_train, y_train)
    print(f"{n_components} components: test MSE = {mean_squared_error(y_test, pcr.predict(X_test)):.1f}")


### Practice

Change the number of PCA components from 2 to 20 in the digits example. Plot cumulative explained variance. Is retaining more variance always the same as improving a downstream prediction task? Explain.

> **YOUR ANSWER:**


## Part 0 — Centering, scaling, and directions of variation

PCA finds directions in which observations vary most. Centering subtracts each feature’s mean; scaling is often important when units differ. For centered $X$, the covariance matrix is

$$C=\frac{1}{n-1}X^TX.$$

Its eigenvectors are principal directions, and its eigenvalues quantify variance along them. SVD finds related directions without first building $C$.


In [ ]:
scaled_digits = StandardScaler().fit_transform(X_digits)
pca_all = PCA().fit(scaled_digits)
cumulative_variance = np.cumsum(pca_all.explained_variance_ratio_)
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance)
plt.axhline(0.90, color="black", linestyle="--", label="90% variance")
plt.xlabel("number of components"); plt.ylabel("cumulative explained variance")
plt.legend(); plt.show()


### Written response 2

Estimate the fewest components that preserve 90% of explained variance. Why can a two-dimensional plot still be useful even if it preserves much less?

> **YOUR ANSWER:**


## Part 3 — Reconstruction and information loss

Projection to fewer components is lossy. `inverse_transform` maps a reduced point back to the original feature space, producing an approximation. Broad structure can remain while fine detail disappears—the same principle behind low-rank image compression.


In [ ]:
scaler = StandardScaler().fit(X_digits)
for n_components in [2, 10, 30]:
    reducer = PCA(n_components=n_components).fit(scaled_digits)
    reconstruction = scaler.inverse_transform(reducer.inverse_transform(reducer.transform(scaled_digits[[0]])))
    plt.imshow(reconstruction.reshape(8, 8), cmap="gray_r")
    plt.title(f"reconstruction: {n_components} PCs"); plt.axis("off"); plt.show()


### Practice and caution

Which details disappear first in the reconstructions? Why is PCA for visualization different from PCA before regression? Remember: PCA does not use $y$, so high-variance directions are not automatically the most predictive. Fit scaling and PCA on training data only to avoid leakage.

> **YOUR ANSWER:**


## Part 4 — Choosing a number of components

There is no universal explained-variance cutoff. Fewer components simplify visualization and can reduce noise; more components preserve more information. Choose the number using the goal: a plot may need two components, while a predictive pipeline should compare candidates with cross-validation.


### Think–pair–share

Suppose one feature is measured in dollars and another is a fraction between 0 and 1. Predict what happens without scaling before PCA. Then explain why fitting the scaler before the train/test split would be data leakage.

> **YOUR ANSWER:**


## Lesson summary

PCA rotates data to orthogonal directions ordered by variation. It can visualize, compress, and pre-process data, but it does not know the response variable or replace evaluation of a downstream task.


## Part 5 — Topic modeling with dimension reduction

The TF–IDF notebook represented each document as a sparse row in a document–term matrix. We can reduce that high-dimensional matrix to a few latent directions. For sparse TF–IDF data, `TruncatedSVD` is usually preferred over ordinary PCA because it works directly with sparse matrices and does not require centering every zero. This approach is often called **latent semantic analysis (LSA)**.

Each latent direction is a possible *topic*: a weighted collection of terms that tend to occur together. Topics are discovered from word co-occurrence, so they need not be perfect labels or represent a single coherent idea.


In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

topic_documents = [
    "The telescope observed a distant galaxy and bright stars.",
    "Astronauts prepare a rocket for an orbit around Earth.",
    "A satellite sends images of planets and space.",
    "Cells use proteins to carry genetic information.",
    "The microscope shows bacteria in a tissue sample.",
    "Scientists study genes, cells, and evolution.",
    "The recipe uses tomatoes, herbs, and olive oil.",
    "Bake the bread until the crust is golden brown.",
    "The chef prepared a warm soup with vegetables.",
]

topic_vectorizer = TfidfVectorizer(stop_words="english")
document_term = topic_vectorizer.fit_transform(topic_documents)
print("document–term matrix shape:", document_term.shape)
pd.DataFrame(document_term.toarray(), columns=topic_vectorizer.get_feature_names_out()).round(2)


### Before running the model

Read the nine documents and propose three human-assigned topic labels. Which words do you expect to be most helpful for distinguishing those groups? Which words might be shared across groups or too general to help?

> **YOUR ANSWER:**


In [ ]:
lsa = TruncatedSVD(n_components=3, random_state=232)
document_topics = lsa.fit_transform(document_term)
terms = topic_vectorizer.get_feature_names_out()

for topic_number, component in enumerate(lsa.components_, start=1):
    top_terms = terms[component.argsort()[-6:]][::-1]
    print(f"Topic {topic_number}: {', '.join(top_terms)}")

print()
print("Explained variance ratio:", lsa.explained_variance_ratio_.round(3))
pd.DataFrame(document_topics, columns=["topic_1", "topic_2", "topic_3"]).round(2)


## Interpreting latent topics

The output names the *terms*, not the topics. You supply a tentative label after inspecting the top terms and the documents with high values on that component. For example, a component whose large terms include `rocket`, `orbit`, and `satellite` may reasonably be called a space topic. A document can have a mixture of topic scores rather than belonging to exactly one topic.


In [ ]:
labels = ["space", "space", "space", "biology", "biology", "biology", "food", "food", "food"]
plt.figure(figsize=(7, 5))
for label in sorted(set(labels)):
    mask = np.array(labels) == label
    plt.scatter(document_topics[mask, 0], document_topics[mask, 1], label=label, s=70)
for index, text in enumerate(topic_documents):
    plt.annotate(str(index + 1), (document_topics[index, 0], document_topics[index, 1]))
plt.xlabel("latent topic direction 1")
plt.ylabel("latent topic direction 2")
plt.legend(title="human label")
plt.show()


### ✏️ Topic-modeling practice

1. Give each discovered component a tentative topic label, using its top terms.
2. Does the two-dimensional plot separate the three human-labeled groups? Identify one document that may be mixed or ambiguous.
3. Change `n_components` to 2 and then 4. What changes in the top terms and explained variance?
4. Why should these automatically discovered topics be reviewed by a person before using them to summarize or make decisions about a corpus?

> **YOUR ANSWER:**


## PCA, SVD, and topic modeling

PCA and truncated SVD share the dimension-reduction idea: find a small number of directions that retain important structure. PCA is commonly applied to centered dense features; truncated SVD is convenient for a sparse document–term matrix. Neither method “understands” language. Results depend on the corpus, preprocessing, number of components, and how people interpret the resulting term lists.
